# Q1. 969. Activation function leaky relu




Like ReLU but allows a small gradient for negative inputs — prevents dead neurons during training.

$$f(x) = \begin{cases} x & x \geq 0 \\ \alpha x & x < 0 \end{cases}$$

**Input**
```
x     = [-2.0, -1.0, 0.0, 3.0]
alpha = 0.1
```
**Expected Output**
```
[-0.2, -0.1, 0.0, 3.0]

-2.0 × 0.1 = -0.2
-1.0 × 0.1 = -0.1
 0.0 → 0.0
 3.0 → 3.0  (positive, unchanged)
```

In [ ]:
import numpy as np

def leaky_relu(x, alpha=0.01):
    """
    Applies the Leaky ReLU activation element-wise.

    Leaky ReLU is defined as:
        f(x) = max(x, alpha * x)

    Args:
        x (np.ndarray): Input values (a 1D vector).
        alpha (float): Negative-slope coefficient.

    Returns:
        np.ndarray: Output vector after applying Leaky ReLU.
    """
    # Apply Leaky ReLU: x if x >= 0 else alpha * x
    return np.where(x >= 0, x, x * alpha)

result = leaky_relu(x, alpha)
result

# Q2. 11. Activation function relu




Zero out all negative values — keeps positive activations, kills negatives.

$$\text{ReLU}(x) = \max(0, x)$$

**Input**
```
x = [-2.0, -0.5, 0.0, 1.5, 3.0]
```
**Expected Output**
```
[0.0, 0.0, 0.0, 1.5, 3.0]

-2.0 → 0.0  (negative)
-0.5 → 0.0  (negative)
 0.0 → 0.0
 1.5 → 1.5  (positive, unchanged)
 3.0 → 3.0  (positive, unchanged)
```

In [4]:
import numpy as np


def relu(x):
    """
    Applies the ReLU activation function elementwise.

    Args:
        x (np.ndarray): Input values.

    Returns:
        np.ndarray: Output array where each element is max(0, x_i).
    """
    # Apply ReLU elementwise
    return np.where(x>=0,x,0)
x = np.array([0.0, 0.0, 0.0, 1.5, 3.0])
result = relu(x)
result

array([0. , 0. , 0. , 1.5, 3. ])

# Q3. 730. Activation function tanh




Squash any real value into $(-1, 1)$ — symmetric around zero, used in RNNs and older networks.

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

**Numerically stable implementation — split by sign to avoid `exp(+inf)` overflow:**
problem:
```
x = 1000
e^1000 = overflow (inf)
inf - inf = nan  ← breaks
```
- but exp(negatives) -> always stayes in (0, 1] 
- so in both the case we convert the exp into negatives then do.
- for For x >= 0, divide numerator and denominator by e^x:
- For x <> 0, divide numerator and denominator by e^-x:
```
x >= 0 → (1 - e^{-2x}) / (1 + e^{-2x})   ← only exp(negative) → stays in (0,1]
x <  0 → (e^{2x} - 1)  / (e^{2x} + 1)    ← only exp(negative) → stays in (0,1]
```

**Input**
```
x = [-1, 0, 1]
```
**Expected Output**
```
[-0.7616, 0.0, 0.7616]

tanh(-1) = (e⁻¹ - e¹) / (e⁻¹ + e¹) = -2.350 / 3.086 ≈ -0.7616
tanh( 0) = (1 - 1) / (1 + 1) = 0.0
tanh( 1) = (e¹ - e⁻¹) / (e¹ + e⁻¹) =  2.350 / 3.086 ≈  0.7616
```

In [ ]:
import numpy as np

def tanh_activation(x):
    """
    Applies the tanh activation element-wise.

    Args:
        x (np.ndarray): A 1D array of real-valued pre-activation outputs.

    Returns:
        np.ndarray: A 1D array where each element is tanh(x_i).
    """
    # Stable implementation using numpy
    # For x >= 0: (1 - exp(-2x)) / (1 + exp(-2x))
    # For x < 0:  (exp(2x) - 1) / (exp(2x) + 1)

    # We can implement this using np.where to select the formula
    # or simply use np.tanh(x) if allowed. 
    # But typically "implement tanh" implies not using the built-in if checking understanding.
    # However, for a NumPy coding interview, usually using np.tanh is the best answer unless "scratch" is specified.
    # The previous solution used math.exp manual implementation.
    # Let's stick to the manual implementation but vectorized for 'educational' purposes if the prompt implies it.
    # But wait, usually simpler is better for "numpy" tag unless "scratch" is strictly requested.
    # The prompt says "No built-in tanh helpers (np/torch/jax)". So I must implement it.

    pos_mask = x >= 0
    neg_mask = ~pos_mask

    result = np.zeros_like(x, dtype=float)

    # Case x >= 0
    z_pos = np.exp(-2.0 * x[pos_mask])
    result[pos_mask] = (1.0 - z_pos) / (1.0 + z_pos)

    # Case x < 0
    z_neg = np.exp(2.0 * x[neg_mask])
    result[neg_mask] = (z_neg - 1.0) / (z_neg + 1.0)

    return result

result = tanh_activation(x)
result

# Q4. 872. Backpropagation through activation




Compute how loss flows back through an activation using the chain rule: $dZ = dA \odot g'(Z)$.

$$\text{ReLU: } g'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases} \qquad \text{Sigmoid: } g'(z) = \sigma(z)(1 - \sigma(z))$$

**Input**
```
dA = [[1.0, -2.0,  3.0],
      [0.5,  1.0, -1.5]]

Z  = [[2.0, -1.0,  0.0],
      [-3.0, 4.0,  1.0]]

activation = "relu"
```
**Expected Output**
```
[[1.0, 0.0,  0.0 ],
 [0.0, 1.0, -1.5 ]]

Z > 0:  [[T, F, F], [F, T, T]]
dZ = dA × mask:
  1.0×1, -2.0×0,  3.0×0  → [1.0,  0.0,  0.0]
  0.5×0,  1.0×1, -1.5×1  → [0.0,  1.0, -1.5]
```

In [6]:
import numpy as np


def activation_backward(dA, Z, activation):
    """
    Computes dZ (gradient w.r.t. pre-activation) given dA.

    Args:
        dA (np.ndarray): Upstream gradient with shape (m, n).
        Z (np.ndarray): Pre-activation values with shape (m, n).
        activation (str): Either "relu" or "sigmoid".

    Returns:
        np.ndarray: dZ with shape (m, n).
    """
    if activation == "relu":
        # ReLU derivative: 1 where z > 0, else 0
        dZ = dA * (Z > 0)
    else:
        # Sigmoid derivative: s * (1 - s)
        s = 1.0 / (1.0 + np.exp(-Z))
        dZ = dA * s * (1.0 - s)

    return dZ
dA = np.array([[1.0, -2.0,  3.0],
      [0.5,  1.0, -1.5]])

Z  = np.array([[2.0, -1.0,  0.0],
      [-3.0, 4.0,  1.0]])

activation = "relu"
result = activation_backward(dA, Z, activation)
result

array([[ 1. , -0. ,  0. ],
       [ 0. ,  1. , -1.5]])

# Q5. 407. Backpropagation through linear layer




Compute gradients for weights, bias, and inputs of a linear layer $Y = XW + b$ given upstream gradient $dY$.

$$dX = dY W^T, \qquad dW = X^T dY, \qquad db = \sum_{i=1}^{n} dY_i$$

**Input**
```
X  = [[1.0, 2.0],    shape (2, 2)
      [3.0, 4.0]]

W  = [[1.0, 0.0],    shape (2, 2)
      [0.0, 1.0]]

dY = [[1.0, 1.0],    shape (2, 2)
      [2.0, 0.0]]
```
**Expected Output**
```
dX = [[1.0, 1.0], [2.0, 0.0]]   = dY @ Wᵀ
dW = [[7.0, 1.0], [10.0, 2.0]]  = Xᵀ @ dY  →  [[1,3],[2,4]] @ [[1,1],[2,0]]
db = [3.0, 1.0]                  = sum(dY, axis=0) = [1+2, 1+0]
```

In [ ]:
import numpy as np

def linear_backward(X, W, b, dY):
    dx = dY @ W.T
    dw = X.T @ dY
    db = dY.sum(axis=0)
    return dx, dw, db

X = np.array([[1.0, 2.0],
              [3.0, 4.0]])

W = np.array([[1.0, 0.0],
              [0.0, 1.0]])

b = np.array([0.5, -0.5])

dY = np.array([[1.0, 1.0],
               [2.0, 0.0]])

result = linear_backward(X, W, b, dY)
result

# Q6. 1324. Adam optimizer step




Adaptive optimizer that maintains per-parameter learning rates using running mean and variance of gradients.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t}, \qquad \theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

**Input**
```
params = [1.0, 2.0],  grads = [0.1, -0.2]
m = [0.0, 0.0],  v = [0.0, 0.0],  t = 1
lr=0.001, beta1=0.9, beta2=0.999
```
**Expected Output**
```
new_params = [0.999, 2.001]
new_m      = [0.01, -0.02]
new_v      = [1e-5, 4e-5]

m = 0.9×0 + 0.1×[0.1,-0.2]       = [0.01, -0.02]
v = 0.999×0 + 0.001×[0.01, 0.04] = [1e-5, 4e-5]

m̂ = [0.01,-0.02] / (1-0.9)  = [0.1, -0.2]
v̂ = [1e-5, 4e-5] / (1-0.999) = [0.01, 0.04]

θ = [1,2] - 0.001 × [0.1,-0.2] / [0.1, 0.2] = [1-0.001, 2+0.001] = [0.999, 2.001]
```

In [ ]:
import numpy as np


def adam_step(params, grads, m, v, t, lr=0.001, beta1=0.9,
              beta2=0.999, eps=1e-8):
    """
    Performs one Adam optimizer update step.

    Args:
        params (list[float]): Current parameters θ (length n).
        grads (list[float]): Current gradients g_t (length n).
        m (list[float]): First moment vector m_{t-1} (length n).
        v (list[float]): Second moment vector v_{t-1} (length n).
        t (int): Time step (1-indexed), used for bias correction.
        lr (float): Learning rate α.
        beta1 (float): Exponential decay for first moment.
        beta2 (float): Exponential decay for second moment.
        eps (float): Small constant for numerical stability.

    Returns:
        tuple[list[float], list[float], list[float]]:
            (new_params, new_m, new_v)
    """
    # Convert inputs to NumPy arrays for vectorized computation
    params_arr = np.array(params, dtype=float)
    grads_arr = np.array(grads, dtype=float)
    m_arr = np.array(m, dtype=float)
    v_arr = np.array(v, dtype=float)

    # Update biased first and second moment estimates
    new_m = beta1 * m_arr + (1.0 - beta1) * grads_arr
    new_v = beta2 * v_arr + (1.0 - beta2) * (grads_arr ** 2)

    # Compute bias-corrected moments
    m_hat = new_m / (1.0 - (beta1 ** t))
    v_hat = new_v / (1.0 - (beta2 ** t))

    # Update parameters using Adam rule
    new_params = params_arr - lr * m_hat / (np.sqrt(v_hat) + eps)

    # Return Python lists
    return new_params.tolist(), new_m.tolist(), new_v.tolist()

result = adam_step(params, grads, m, v, t, lr, beta1, beta2, eps)
result

# Q7. 1343. Batch normalization forward pass




Normalize each feature across the batch to zero mean and unit variance, then rescale with learnable $\gamma, \beta$.

$$\mu_j = \frac{1}{m}\sum_i X_{ij}, \quad \sigma^2_j = \frac{1}{m}\sum_i (X_{ij}-\mu_j)^2, \quad \hat{X}_{ij} = \frac{X_{ij}-\mu_j}{\sqrt{\sigma^2_j+\epsilon}}, \quad Y_{ij} = \gamma_j \hat{X}_{ij} + \beta_j$$

**Input**
```
X     = [[1.0, 2.0],
         [3.0, 4.0]]
gamma = [1.0, 1.5],  beta = [0.0, 0.5]
```
**Expected Output**
```
[[-1.0, -1.0],
 [ 1.0,  2.0]]

feature 0: μ=2, σ²=1  → X̂=[-1, 1] → Y = 1.0×[-1,1] + 0.0 = [-1.0,  1.0]
feature 1: μ=3, σ²=1  → X̂=[-1, 1] → Y = 1.5×[-1,1] + 0.5 = [-1.0,  2.0]

row 0: [-1.0, -1.0]
row 1: [ 1.0,  2.0]
```

In [ ]:
import numpy as np


def batch_norm_forward(X, gamma, beta, eps=1e-5):
    """
    Performs the forward pass of batch normalization (training-time).

    BatchNorm per feature j is:
        mu_j = (1/m) * sum_i X_ij
        var_j = (1/m) * sum_i (X_ij - mu_j)^2
        Xhat_ij = (X_ij - mu_j) / sqrt(var_j + eps)
        Y_ij = gamma_j * Xhat_ij + beta_j

    Args:
        X (np.ndarray): Input batch of shape (m, d).
        gamma (np.ndarray): Scale parameters of length d.
        beta (np.ndarray): Shift parameters of length d.
        eps (float): Small constant for numerical stability.

    Returns:
        np.ndarray: Batch-normalized output Y of shape (m, d).
    """
    # Compute mean and variance per feature (column)
    mean = np.mean(X, axis=0)
    var = np.mean((X - mean) ** 2, axis=0)

    # Normalize and apply scale + shift
    x_hat = (X - mean) / np.sqrt(var + eps)
    y_arr = gamma * x_hat + beta

    # Return as NumPy array
    return y_arr

X= np.array([[1.0, 2.0],
         [3.0, 4.0]])
gamma = [1.0, 1.5]
beta = [0.0, 0.5]

result = batch_norm_forward(X, gamma, beta)
result

array([[-0.999995 , -0.9999925],
       [ 0.999995 ,  1.9999925]])

# Q8. 943. Gradient checking



Verify backprop gradients are correct by comparing them against numerical gradients from finite differences — relative error should be ~$10^{-7}$ or smaller.

**Forward pass:**
$$z = XW + b, \quad a = \sigma(z), \quad L = -\frac{1}{n}\sum\left[y\log a + (1-y)\log(1-a)\right]$$

**Numerical gradient (centered difference):**
$$g_{\text{num}}(i) = \frac{L(W_i + \epsilon) - L(W_i - \epsilon)}{2\epsilon}$$

**Relative error:**
$$\text{rel\_err} = \frac{|g_{\text{anal}} - g_{\text{num}}|}{\max(10^{-8},\ |g_{\text{anal}}| + |g_{\text{num}}|)}$$

**Input**
```
X = [[0.2, -0.1],    y = [1.0, 0.0]
     [1.0,  0.3]]

W = [[0.05], [-0.02]],  b = [0.01]
```
**Expected Output**
```
[[1.2e-07],
 [8.5e-08]]

← tiny errors confirm analytical gradients match numerical ones
← if error > 1e-3, backprop has a bug
```

In [ ]:
import numpy as np

def gradient_check_dense(X, y, W, b, eps=1e-5):
    """
    Performs gradient checking for a single dense layer + sigmoid + BCE.
    """
    # Ensure y is (n, 1) for broadcasting
    # The input y might be (n,) or (n, 1) depending on how it's passed
    y = y.reshape(-1, 1)

    # Ensure b is (1,) or scalar
    b = b.reshape(1,)

    # Numerically stable sigmoid
    def sigmoid(z):
        pos_mask = z >= 0
        neg_mask = ~pos_mask
        out = np.empty_like(z)
        out[pos_mask] = 1.0 / (1.0 + np.exp(-z[pos_mask]))
        exp_z = np.exp(z[neg_mask])
        out[neg_mask] = exp_z / (1.0 + exp_z)
        return out

    # Forward pass and loss computation
    def forward_loss(W_local):
        z = X @ W_local + b
        a = sigmoid(z)
        a = np.clip(a, 1e-12, 1.0 - 1e-12)
        loss = -np.mean(
            y * np.log(a) + (1.0 - y) * np.log(1.0 - a)
        )
        return loss

    # Analytical gradients
    z = X @ W + b
    a = sigmoid(z)
    a = np.clip(a, 1e-12, 1.0 - 1e-12)
    n = X.shape[0]
    dz = (a - y) / n
    grad_anal = X.T @ dz

    # Numerical gradients
    grad_num = np.zeros_like(W)
    for i in range(W.shape[0]):
        W_pos = W.copy()
        W_neg = W.copy()
        W_pos[i, 0] += eps
        W_neg[i, 0] -= eps
        loss_pos = forward_loss(W_pos)
        loss_neg = forward_loss(W_neg)
        grad_num[i, 0] = (loss_pos - loss_neg) / (2.0 * eps)

    # Relative error
    rel_err = np.abs(grad_anal - grad_num)
    denom = np.maximum(1e-8, np.abs(grad_anal) + np.abs(grad_num))
    rel_err = rel_err / denom
    return rel_err

result = gradient_check_dense(X, y, W, b, eps)
result